In [ ]:
!pip install openai
!pip install groq


In [ ]:
import os
from openai import OpenAI
from groq import Groq

In [ ]:

# Set your Groq API Key
groq_api_key = "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"
os.environ["GROQ_API_KEY"] = groq_api_key

# Create client
client = Groq(api_key=groq_api_key)

def prompt_based_query(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # or another Groq-supported model
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_query}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# Example
answer = prompt_based_query("What is Machine Learning?")
print(answer)

In [ ]:
!pip install langchain faiss-cpu openai langchain_community tiktoken

In [ ]:
from langchain_community.document_loaders import TextLoader

# Load the text file
loader = TextLoader("/content/medical_company_data.txt", encoding="utf-8")

# Read the document
documents = loader.load()

# Print content
print(documents[0].page_content)

In [ ]:
!pip install -q langchain-huggingface sentence-transformers faiss-cpu

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(documents, embeddings)

In [ ]:
from groq import Groq

client = Groq(api_key='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')

In [ ]:
def rag_query(query):
    # Retrieve relevant chunks
    docs = vector_db.similarity_search(query, k=3)

    # Build context
    context = "\n\n".join(doc.page_content for doc in docs)

    # Generate answer using Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Use only the provided context to answer the question. If the answer is not in the context, say 'I don't know based on the provided document.'"
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
print(rag_query("What is our company Prescription policy?"))

In [ ]:
!pip install transformers datasets bitsandbytes peft accelerate

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb
model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer=AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Set pad_token for causal language modeling
#define the bitsandbytes config for 8 bit quantization
quantization_config=BitsAndBytesConfig(load_in_8bit=True)
#load the model
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
    )

In [ ]:
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training
model= prepare_model_for_kbit_training(model)
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)
# Enable input gradients for the PeftModel (sometimes needed explicitly for Trainer compatibility)
model.enable_input_require_grads()
model

In [ ]:
from datasets import Dataset
data=[
    {"text":"Q:What is Appointment Policy?\nA:Appointments can be booked online via the website or mobile app,Patients must book at least 30 minutes in advance for same-day consultations.Appointments can be rescheduled or cancelled up to 15 minutes before the scheduled time.A full refund is issued for cancellations made at least 1 hour before the appointment."}
]
dataset = Dataset.from_list(data)

In [ ]:
#Step-4: Tokenization
def tokenize_function(example):
  return tokenizer(example["text"],truncation=True, padding="max_length")

tokenized_datasets = dataset.map(tokenize_function)

In [ ]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_datasets=tokenized_datasets.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets
)
trainer.train()

In [ ]:
print(rag_query("What is Appointment Policy? "))